In [1]:
import os
print(os.getcwd())

C:\Users\palak\OneDrive\Documents\f1-race-intelligence-platform\notebooks


In [5]:
path = '../data/raw/f1-data'

races = pd.read_csv(f'{path}/races.csv', na_values='\\N')
results = pd.read_csv(f'{path}/results.csv', na_values='\\N')
drivers = pd.read_csv(f'{path}/drivers.csv', na_values='\\N')
constructors = pd.read_csv(f'{path}/constructors.csv', na_values='\\N')
status = pd.read_csv(f'{path}/status.csv', na_values='\\N')

for name, df in [('races', races), ('results', results), ('drivers', drivers), 
                  ('constructors', constructors), ('status', status)]:
    print(name, df.shape)

races (1125, 18)
results (26759, 18)
drivers (861, 9)
constructors (212, 5)
status (139, 2)


In [6]:
for name, df in [('races', races), ('results', results), ('drivers', drivers), 
                  ('constructors', constructors), ('status', status)]:
    print(f"--- {name} ---")
    print(df.isnull().sum()[df.isnull().sum() > 0])
    print()

--- races ---
time            731
fp1_date       1035
fp1_time       1057
fp2_date       1035
fp2_time       1057
fp3_date       1053
fp3_time       1072
quali_date     1035
quali_time     1057
sprint_date    1107
sprint_time    1110
dtype: int64

--- results ---
number                 6
position           10953
time               19079
milliseconds       19079
fastestLap         18507
rank               18249
fastestLapTime     18507
fastestLapSpeed    18507
dtype: int64

--- drivers ---
number    802
code      757
dtype: int64

--- constructors ---
Series([], dtype: int64)

--- status ---
Series([], dtype: int64)



In [7]:
df = (results
      .merge(races[['raceId','year','round','circuitId','name','date']], on='raceId')
      .merge(drivers[['driverId','driverRef','forename','surname']], on='driverId')
      .merge(constructors[['constructorId','name']], on='constructorId', suffixes=('_race','_team'))
      .merge(status, on='statusId'))

print(df.shape)
df.head()

(26759, 28)


,resultId,raceId,driverId,constructorId,number,grid,position,positionText,positionOrder,points,...,year,round,circuitId,name_race,date,driverRef,forename,surname,name_team,status
0,1,18,1,1,22.0,1,1.0,1,1,10.0,...,2008,1,1,Australian Grand Prix,2008-03-16,hamilton,Lewis,Hamilton,McLaren,Finished
1,2,18,2,2,3.0,5,2.0,2,2,8.0,...,2008,1,1,Australian Grand Prix,2008-03-16,heidfeld,Nick,Heidfeld,BMW Sauber,Finished
2,3,18,3,3,7.0,7,3.0,3,3,6.0,...,2008,1,1,Australian Grand Prix,2008-03-16,rosberg,Nico,Rosberg,Williams,Finished
3,4,18,4,4,5.0,11,4.0,4,4,5.0,...,2008,1,1,Australian Grand Prix,2008-03-16,alonso,Fernando,Alonso,Renault,Finished
4,5,18,5,1,23.0,3,5.0,5,5,4.0,...,2008,1,1,Australian Grand Prix,2008-03-16,kovalainen,Heikki,Kovalainen,McLaren,Finished


In [8]:
df['position'] = pd.to_numeric(df['position'], errors='coerce')  # DNFs become NaN instead of crashing
df['grid'] = pd.to_numeric(df['grid'], errors='coerce')
df['points'] = pd.to_numeric(df['points'], errors='coerce')
df['finished'] = df['status'] == 'Finished'

print(df[['position','grid','points','finished']].describe())
print(df['finished'].value_counts())

           position          grid        points
count  15806.000000  26759.000000  26759.000000
mean       8.020499     11.134796      1.987632
std        4.840796      7.202860      4.351209
min        1.000000      0.000000      0.000000
25%        4.000000      5.000000      0.000000
50%        8.000000     11.000000      0.000000
75%       11.000000     17.000000      2.000000
max       33.000000     34.000000     50.000000
finished
False    19085
True      7674
Name: count, dtype: int64


In [9]:
df['winner'] = (df['position'] == 1).astype(int)

print(df['winner'].value_counts())

winner
0    25631
1     1128
Name: count, dtype: int64


In [10]:
df_modern = df[df['year'] >= 2014].copy()
print(df_modern.shape)
print(df_modern['winner'].value_counts())

(4626, 30)
winner
0    4398
1     228
Name: count, dtype: int64


In [11]:
import os
os.makedirs('../data/processed', exist_ok=True)

df.to_csv('../data/processed/master_full.csv', index=False)
df_modern.to_csv('../data/processed/master_modern.csv', index=False)

print("Saved successfully")

Saved successfully
